# Part 2: Multi-Source RAG with Routing

RAG system that routes queries to CSV (sales data), unstructured text (product pages), or both.

In [ ]:
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.part2_multi_source import answer_question
from src.part2_router import route_question, SourceRoute
from src.part2_retrievers import load_csv, load_product_pages, build_bm25_index

CSV_PATH = PROJECT_ROOT / "data" / "structured" / "daily_sales.csv"
TEXT_DIR = PROJECT_ROOT / "data" / "unstructured"
assert CSV_PATH.exists(), f"CSV not found: {CSV_PATH}"
assert TEXT_DIR.exists(), f"Text dir not found: {TEXT_DIR}"

In [ ]:
# Pre-load data for efficiency
df = load_csv(CSV_PATH)
pages = load_product_pages(TEXT_DIR)
bm25, _ = build_bm25_index(pages)
print(f"Loaded {len(df)} sales records, {len(pages)} product pages")

In [ ]:
TEST_QUESTIONS = [
    (1, "What was the total revenue for Electronics category in December 2024?", "CSV Only"),
    (2, "Which region had the highest sales volume?", "CSV Only"),
    (3, "What are the key features of the Wireless Bluetooth Headphones?", "Text Only"),
    (4, "What do customers say about the Air Fryer's ease of cleaning?", "Text Only"),
    (5, "Which product has the best customer reviews and how well is it selling?", "Both"),
    (6, "I want a product for fitness that is highly rated and sells well in the West region. What do you recommend?", "Both"),
]

In [ ]:
results = []
for num, question, expected in TEST_QUESTIONS:
    print(f"\n{'='*60}")
    print(f"Q{num} (expected: {expected}): {question}")
    print("="*60)
    route = route_question(question)
    print(f"Routed to: {route.value}")
    answer = answer_question(question, CSV_PATH, TEXT_DIR, df=df, pages=pages, bm25=bm25)
    print(f"\nAnswer:\n{answer}")
    results.append((num, question, expected, answer))
    time.sleep(15)  # Avoid Groq rate limit

In [ ]:
output_path = PROJECT_ROOT / "part2_results.txt"
with open(output_path, "w") as f:
    for num, question, expected, answer in results:
        f.write(f"Question {num} (expected: {expected}): {question}\n")
        f.write(f"Answer:\n{answer}\n")
        f.write("\n" + "-"*60 + "\n")
print(f"Results saved to {output_path}")